Import Datasets for BDC from WRDS


This file is used to extract data from WRDS
1. daily and monthly trade data = crsp
2. quarterly nav = compustat
3. interest rates and credit spread = frb (federal reserve board on wrds)

Using the files above, create df_event_study

In [2]:
import wrds
import pandas as pd
import numpy as np
import duckdb as dd
import statsmodels.formula.api as smf
db = wrds.Connection()

Loading library list...
Done


In [3]:
# db.list_libraries()
# # db.list_tables('crsp_q_mutualfunds')
# # db.describe_table('crsp_q_mutualfunds','monthly_returns')
# db.list_tables('crsp_a_stock')
# db.describe_table('crsp_a_stock','msf')

# db.list_libraries()

In [4]:
# 1. df_trade monthly

query = """SELECT
cast((EXTRACT(YEAR FROM date) * 100 + EXTRACT(MONTH FROM date)) as int) AS yyyymm,
    cast(a.date as date) as trade_date,
    a.permno,
    b.ticker,
    ABS(a.prc) AS price,
    a.ret,
    a.vol,
    a.shrout
FROM crsp_a_stock.msf AS a
JOIN crsp_a_stock.dsenames AS b
    ON a.permno = b.permno
    AND a.date BETWEEN b.namedt AND b.nameendt
WHERE b.ticker IN ('ARCC', 'MAIN')
    AND a.date >= '2015-01-01'
ORDER BY b.ticker, a.date
"""
df_trade_monthly = db.raw_sql(query)
df_trade_monthly.head(10)

,yyyymm,trade_date,permno,ticker,price,ret,vol,shrout
0,201501,2015-01-30,90401,ARCC,16.65,0.066966,499573.0,314108.0
1,201502,2015-02-27,90401,ARCC,17.3,0.039039,386332.0,314108.0
2,201503,2015-03-31,90401,ARCC,17.17,0.017341,452024.0,314469.0
3,201504,2015-04-30,90401,ARCC,17.02,-0.008736,378093.0,314469.0
4,201505,2015-05-29,90401,ARCC,16.75,-0.015864,358205.0,314469.0
5,201506,2015-06-30,90401,ARCC,16.46,0.005373,335775.0,314469.0
6,201507,2015-07-31,90401,ARCC,16.09,-0.022479,262802.0,314469.0
7,201508,2015-08-31,90401,ARCC,15.77,-0.019888,384468.0,314469.0
8,201509,2015-09-30,90401,ARCC,14.48,-0.057705,435538.0,314469.0
9,201510,2015-10-30,90401,ARCC,15.23,0.051796,319545.0,314469.0


In [5]:
# 2. df_trade daily
query = """
SELECT 
    d.date as trade_date,
    d.permno,
    n.ticker,
    d.prc AS price,
    d.ret,
    d.vol,
    d.shrout
FROM crsp.dsf d
JOIN crsp.dsenames n
    ON d.permno = n.permno
WHERE n.ticker IN ('ARCC','MAIN')
  AND d.date BETWEEN n.namedt AND n.nameendt
  AND d.date >= '2015-12-01'
ORDER BY d.permno, d.date"""

df_trade_daily = db.raw_sql(query)
df_trade_daily.head()


,trade_date,permno,ticker,price,ret,vol,shrout
0,2015-12-01,90401,ARCC,15.87,0.003161,1425885.0,314469.0
1,2015-12-02,90401,ARCC,15.73,-0.008822,1233942.0,314469.0
2,2015-12-03,90401,ARCC,15.68,-0.003179,1548635.0,314469.0
3,2015-12-04,90401,ARCC,15.76,0.005102,1454772.0,314469.0
4,2015-12-07,90401,ARCC,15.36,-0.025381,1968566.0,314469.0


In [6]:
# 3. df_rate (monthly)

query = """
SELECT
    CAST(EXTRACT(YEAR FROM date) * 100 + EXTRACT(MONTH FROM date) AS INT) AS yyyymm,
    date AS rate_date,
    gs10 AS dgs10,
    baa,
    aaa
FROM frb.rates_monthly
WHERE date >= '2015-01-01'
"""

# Use .raw_sql() instead of .raw_query()
df_rate = db.raw_sql(query)

# Calculate spread if needed (baa - aaa)
df_rate.loc[:, 'spread'] = df_rate['baa'] - df_rate['dgs10']

df_rate.head()

,yyyymm,rate_date,dgs10,baa,aaa,spread
0,201501,2015-01-31,1.88,4.45,3.46,2.57
1,201502,2015-02-28,1.98,4.51,3.61,2.53
2,201503,2015-03-31,2.04,4.54,3.64,2.5
3,201504,2015-04-30,1.94,4.48,3.52,2.54
4,201505,2015-05-31,2.2,4.89,3.98,2.69


In [7]:
# 4. df_rate (daily)

query = """
SELECT 
    CAST(EXTRACT(YEAR FROM date) * 100 + EXTRACT(MONTH FROM date) AS INT) AS yyyymm,
    date AS rate_date,
    dgs10,  
    dbaa, 
    daaa,
    (dbaa - dgs10) AS spread
FROM frb.rates_daily
WHERE cast(date as date) >= '2016-01-01'
ORDER BY date
"""

df_rate_daily = db.raw_sql(query)
df_rate_daily.head()

,yyyymm,rate_date,dgs10,dbaa,daaa,spread
0,201601,2016-01-01,<NA>,<NA>,<NA>,<NA>
1,201601,2016-01-02,<NA>,<NA>,<NA>,<NA>
2,201601,2016-01-03,<NA>,<NA>,<NA>,<NA>
3,201601,2016-01-04,2.24,5.48,4.03,3.24
4,201601,2016-01-05,2.25,5.5,4.02,3.25


In [8]:
# 5. df nav (quarterly) calculate nav_ret

query = """SELECT
tic as ticker,
    gvkey,
    datadate,
    rdq,
    seqq,
    cshoq,
    (seqq / cshoq) AS nav
FROM comp.fundq
WHERE tic in  ('MAIN', 'ARCC')
AND datadate >= '2015-06-01'
ORDER BY datadate;"""

df_nav = db.raw_sql(query).copy()
df_nav.loc[:,"nav_date"] = pd.to_datetime(df_nav["datadate"], errors="coerce")
df_nav.loc[:,"announcement_date"] = pd.to_datetime(df_nav["rdq"], errors="coerce")

# compute nav change

# --------------------------------------------------
# 1) Sort properly (CRITICAL)
# --------------------------------------------------
df_nav = df_nav.sort_values(['ticker', 'announcement_date'])

# --------------------------------------------------
# 2) Compute NAV pct change
# --------------------------------------------------
df_nav.loc[:, 'nav_ret'] = df_nav.groupby('ticker')['nav'].pct_change()

# --------------------------------------------------
# 3) Inspect
# --------------------------------------------------
df_nav[['ticker','announcement_date','nav','nav_ret']]

df_nav.head()

,ticker,gvkey,datadate,rdq,seqq,cshoq,nav,nav_date,announcement_date,nav_ret
1,ARCC,161966,2015-06-30,2015-08-04,5282.441,314.469,16.797971,2015-06-30,2015-08-04,<NA>
3,ARCC,161966,2015-09-30,2015-11-04,5279.802,314.469,16.789579,2015-09-30,2015-11-04,-0.0005
4,ARCC,161966,2015-12-31,2016-02-24,5173.332,314.347,16.457393,2015-12-31,2016-02-24,-0.019785
6,ARCC,161966,2016-03-31,2016-05-04,5179.944,313.954,16.499054,2016-03-31,2016-05-04,0.002531
8,ARCC,161966,2016-06-30,2016-08-03,5218.041,313.954,16.6204,2016-06-30,2016-08-03,0.007355


In [9]:
# convert date string to datetime

df_trade_monthly.loc[:,'trade_date_1'] = pd.to_datetime(df_trade_monthly['trade_date'])
df_trade_daily.loc[:,'trade_date_1'] = pd.to_datetime(df_trade_daily['trade_date'])
df_nav.loc[:,'announcement_date_1'] = pd.to_datetime(df_nav['announcement_date'])
df_rate.loc[:,'rate_date_1'] = pd.to_datetime(df_rate['rate_date'])
df_rate_daily.loc[:,'rate_date_1'] = pd.to_datetime(df_rate_daily['rate_date'])

df_trade_monthly = df_trade_monthly.drop(columns=["trade_date"]).rename(columns={"trade_date_1": "trade_date"})
df_trade_daily = df_trade_daily.drop(columns=["trade_date"]).rename(columns={"trade_date_1": "trade_date"})
df_nav = df_nav.drop(columns=["announcement_date"]).rename(columns={"announcement_date_1": "announcement_date"})
df_rate = df_rate.drop(columns=["rate_date"]).rename(columns={"rate_date_1": "rate_date"})
df_rate_daily = df_rate_daily.drop(columns=["rate_date"]).rename(columns={"rate_date_1": "rate_date"})

df_trade_monthly.info()
df_trade_daily.info()
df_nav.info()
df_rate.info()
df_rate_daily.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   yyyymm      240 non-null    Int64         
 1   permno      240 non-null    Int64         
 2   ticker      240 non-null    string        
 3   price       240 non-null    Float64       
 4   ret         240 non-null    Float64       
 5   vol         240 non-null    Float64       
 6   shrout      240 non-null    Float64       
 7   trade_date  240 non-null    datetime64[ns]
dtypes: Float64(4), Int64(2), datetime64[ns](1), string(1)
memory usage: 16.5 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4572 entries, 0 to 4571
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   permno      4572 non-null   Int64         
 1   ticker      4572 non-null   string        
 2   price       4572 non-null   Floa

In [10]:
# df_trade_nav_monthly (not needed for now)
# ******* NOTE : ASOF join uses the last known NAV until a new NAV is realized (announcement_date)*****

query = """
SELECT 
    dt.ticker,
    dt.trade_date,
    dt.price,
    dt.ret,
    dn.nav ,
    dn.announcement_date,
    dn.nav_ret
FROM df_trade_monthly dt
ASOF JOIN df_nav dn 
    ON UPPER(dt.ticker) = UPPER(dn.ticker) 
   AND dt.trade_date >= dn.announcement_date
where extract(year from dt.trade_date) >= 2016
ORDER BY dt.trade_date
"""

df_trade_nav_monthly = dd.query(query).to_df()

df_trade_nav_monthly.head()


,ticker,trade_date,price,ret,nav,announcement_date,nav_ret
0,ARCC,2016-01-29,13.90,-0.024561,16.789579,2015-11-04,-0.000500
1,MAIN,2016-01-29,28.89,-0.000344,21.785199,2015-11-05,-0.002645
2,ARCC,2016-02-29,13.66,-0.017266,16.457393,2016-02-24,-0.019785
3,MAIN,2016-02-29,29.42,0.024576,21.241996,2016-02-25,-0.024935
4,ARCC,2016-03-31,14.84,0.114202,16.457393,2016-02-24,-0.019785


In [ ]:
# df_trade_nav_daily


query = """
SELECT 
    dt.ticker,
    dt.trade_date,
    dt.price,
    dt.ret,
    dn.nav ,
    dn.announcement_date,
    dn.nav_ret
FROM df_trade_daily dt
ASOF JOIN df_nav dn 
    ON UPPER(dt.ticker) = UPPER(dn.ticker) 
   AND dt.trade_date >= dn.announcement_date
where extract(year from dt.trade_date) >= 2016
ORDER BY dt.ticker, dt.trade_date
"""


df_trade_nav_daily=dd.query(query).to_df()

df_trade_nav_daily.head()

,ticker,trade_date,price,ret,nav,announcement_date,nav_ret
0,ARCC,2016-01-04,14.46,0.014737,16.789579,2015-11-04,-0.0005
1,ARCC,2016-01-05,14.59,0.008990,16.789579,2015-11-04,-0.0005
2,ARCC,2016-01-06,14.54,-0.003427,16.789579,2015-11-04,-0.0005
3,ARCC,2016-01-07,14.08,-0.031637,16.789579,2015-11-04,-0.0005
4,ARCC,2016-01-08,13.91,-0.012074,16.789579,2015-11-04,-0.0005


In [12]:
# ##########################    event study dataset --- pre-event study dataset ######################################
# ******* NOTE : envelops trade records around the announcement date (announcement_date)*****

df_event_study = dd.query("""
    SELECT *
    FROM df_nav
    INNER JOIN df_trade_daily
        ON df_nav.ticker = df_trade_daily.ticker
       AND df_trade_daily.trade_date >= df_nav.announcement_date - INTERVAL '3 day'
       AND df_trade_daily.trade_date <= df_nav.announcement_date + INTERVAL '3 day'
""").to_df()

df_event_study


,ticker,gvkey,datadate,rdq,seqq,cshoq,nav,nav_date,nav_ret,announcement_date,permno,ticker_1,price,ret,vol,shrout,trade_date
0,ARCC,161966,2023-09-30,2023-10-24,10815.000,569.000,19.007030,2023-09-30,0.022495,2023-10-24,90401,ARCC,18.81,0.001064,5049117.0,569000.0,2023-10-23
1,ARCC,161966,2023-09-30,2023-10-24,10815.000,569.000,19.007030,2023-09-30,0.022495,2023-10-24,90401,ARCC,18.99,0.009569,4099140.0,569437.0,2023-10-24
2,ARCC,161966,2023-09-30,2023-10-24,10815.000,569.000,19.007030,2023-09-30,0.022495,2023-10-24,90401,ARCC,18.90,-0.004739,2810351.0,569437.0,2023-10-25
3,ARCC,161966,2023-09-30,2023-10-24,10815.000,569.000,19.007030,2023-09-30,0.022495,2023-10-24,90401,ARCC,18.89,-0.000529,2285730.0,569437.0,2023-10-26
4,ARCC,161966,2023-09-30,2023-10-24,10815.000,569.000,19.007030,2023-09-30,0.022495,2023-10-24,90401,ARCC,18.66,-0.012176,3004359.0,569437.0,2023-10-27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,MAIN,178529,2023-03-31,2023-05-04,2172.922,79.766,27.241206,2023-03-31,0.013691,2023-05-04,92309,MAIN,40.33,0.021022,406613.0,79550.0,2023-05-05
350,MAIN,178529,2022-12-31,2023-02-23,2108.586,78.464,26.873292,2022-12-31,0.033920,2023-02-23,92309,MAIN,39.36,-0.022840,354156.0,77254.0,2023-02-21
351,MAIN,178529,2022-12-31,2023-02-23,2108.586,78.464,26.873292,2022-12-31,0.033920,2023-02-23,92309,MAIN,39.81,0.011433,243010.0,77254.0,2023-02-22
352,MAIN,178529,2022-12-31,2023-02-23,2108.586,78.464,26.873292,2022-12-31,0.033920,2023-02-23,92309,MAIN,40.19,0.009545,306558.0,77254.0,2023-02-23


We analysize df_trade_nav

In [13]:
# trade daily + nav event EVENT STUDY

# 1. Select only necessary columns
# Note: 'ticker' and 'ticker_1' are duplicates from the join; we keep one.

cols_to_keep = [
    'ticker', 
    'announcement_date', 
    'trade_date', 
    'nav', 
    'nav_ret', 
    'ret', 
    'price'
]

df_event_study = df_event_study[cols_to_keep].copy()

# # 2. Recalculate days_from_event for consistent indexing
df_event_study.loc[:, 'days_from_event'] = (pd.to_datetime(df_event_study['trade_date']) - 
                                     pd.to_datetime(df_event_study['announcement_date'])).dt.days

# # 3. Sort for logical flow
# df_event_study = df_event_study.sort_values(['ticker', 'announcement_date', 'trade_date'])

# # 4. Calculate Cumulative Return starting from the beginning of the [-3, +3] window
# df_event_study.loc[:, 'cum_ret'] = df_event_study.groupby(['ticker', 'announcement_date'])['ret'].transform(lambda x: (1 + x).cumprod() - 1)

# # df_event_study.head()

Interpretation:

NAV is stale and it does not explain return which is what we expected.
    We could include only 

In [14]:
# only event records since day -1. ** This dataset used for event study** this is a subset of df_event_study. #

df_reg_daily_event = dd.query("""select 
* from df_event_study
where days_from_event >= -1
order by ticker, announcement_date, days_from_event
""").to_df().head(10)

df_reg_daily_event.head()

,ticker,announcement_date,trade_date,nav,nav_ret,ret,price,days_from_event
0,ARCC,2016-02-24,2016-02-23,16.457393,-0.019785,0.007015,12.92,-1
1,ARCC,2016-02-24,2016-02-24,16.457393,-0.019785,0.000000,12.92,0
2,ARCC,2016-02-24,2016-02-25,16.457393,-0.019785,0.036378,13.39,1
3,ARCC,2016-02-24,2016-02-26,16.457393,-0.019785,0.008962,13.51,2
4,ARCC,2016-05-04,2016-05-03,16.499054,0.002531,-0.006588,15.08,-1


In [15]:
# regression event study

import statsmodels.api as sm
import pandas as pd
import numpy as np

# 1. Ensure the dataset is clean and dummies are integers
# Using drop_first=False because we manually select days, and T-1 is our excluded baseline
df_reg = pd.get_dummies(df_event_study, columns=['days_from_event'], prefix='day')

# 2. Define features (T=0, T=1, T=2)
X_cols = ['day_0', 'day_1', 'day_2']
X = df_reg[X_cols].astype(float) # Force to float to fix the ValueError
X = sm.add_constant(X)

# 3. Define target and force to float
y = pd.to_numeric(df_reg['ret'], errors='coerce').astype(float)

# 4. Handle any potential NaNs created by coerce or existing in data
# 'missing=drop' in OLS handles this, but statsmodels prefers clean arrays
valid_idx = y.notna()
y = y[valid_idx]
X = X[valid_idx]

# 5. Run the OLS
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     1.234
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.297
Time:                        20:05:38   Log-Likelihood:                 946.56
No. Observations:                 354   AIC:                            -1885.
Df Residuals:                     350   BIC:                            -1870.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0005      0.001      0.412      0.6

interpretation:
Day -1 return :- Average return on the day before announcement is negligible .07% (Not signnificant)
Day 0 return :- Average return on the day of announcement is .22% (Not significant)
Day 1 return :- Average return jumps by .46% or 46 basis points.
Mean revision :- 0.0%

Is Day 1 return .46% a NAV surprise?
Regress market return (t+1) = nav_ret

Is it the nav_change that caused the jump on return by the .46% i.e. 46 basis points.

In [16]:
# 1.regress return on day 1 nav_ret

df_day1 = df_event_study[df_event_study['days_from_event'] == 1].copy()

# 2. Define features: The NAV Surprise
X = df_day1['nav_ret'].astype(float)
X = sm.add_constant(X)

# 3. Define target: The Day 1 Return
y = df_day1['ret'].astype(float)

# 4. Run the OLS
model_sensitivity = sm.OLS(y, X, missing='drop').fit()

print(model_sensitivity.summary())

                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     1.685
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.199
Time:                        20:05:38   Log-Likelihood:                 182.48
No. Observations:                  72   AIC:                            -361.0
Df Residuals:                      70   BIC:                            -356.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0056      0.002      2.365      0.0

Interpretation:
    
    1. alpha on day 1 is .63% => book value is does not matter.
    2. NAV is -.14%
    
    We conclude that NAV has no bearing on the price.


In [17]:
# data integrity #

# 1. Check for Look-Ahead Bias (Did we match with a future NAV?)
future_matches = df_reg[df_reg['trade_date'] < df_reg['announcement_date']]

# 2. Check for missing months per ticker since 2016
# Expecting 108 months per ticker (9 years * 12 months)
obs_count = df_reg.groupby('ticker')['trade_date'].count()

# 3. Check for Nulls in key regression columns
nulls = df_reg[['price', 'nav', 'ret', 'nav_ret']].isnull().sum()

print("--- DATA INTEGRITY TEST RESULTS ---")
print(f"Look-ahead bias records: {len(future_matches)}")
print("\nObservations per Ticker (Expected 108):")
print(obs_count)
print("\nMissing Values in Regression Columns:")
print(nulls)

if len(future_matches) == 0 and (obs_count == 108).all():
    print("\n✅ SUCCESS: No records dropped and date logic is sound.")
else:
    print("\n⚠️ WARNING: Check for missing months or join errors.")

--- DATA INTEGRITY TEST RESULTS ---
Look-ahead bias records: 157

Observations per Ticker (Expected 108):
ticker
ARCC    179
MAIN    175
Name: trade_date, dtype: int64

Missing Values in Regression Columns:
price      0
nav        0
ret        0
nav_ret    0
dtype: int64

⚠️ WARNING: Check for missing months or join errors.




In this sectionm we add 
1. federal reserve 10 year tbill rate
2. credit spread 


In [18]:
df_rate_daily.to_csv('data/df_rate_daily.csv', index=False)
df_nav.to_csv('data/df_nav.csv', index=False)
df_trade_daily.to_csv('data/df_trade_daily.csv', index=False)
df_event_study.to_csv('data/df_event_study.csv', index=False)

In [19]:
df_trade_nav_daily.to_csv('data/df_trade_nav_daily.csv', index=False)

In [20]:
df_event_study

,ticker,announcement_date,trade_date,nav,nav_ret,ret,price,days_from_event
0,ARCC,2023-10-24,2023-10-23,19.007030,0.022495,0.001064,18.81,-1
1,ARCC,2023-10-24,2023-10-24,19.007030,0.022495,0.009569,18.99,0
2,ARCC,2023-10-24,2023-10-25,19.007030,0.022495,-0.004739,18.90,1
3,ARCC,2023-10-24,2023-10-26,19.007030,0.022495,-0.000529,18.89,2
4,ARCC,2023-10-24,2023-10-27,19.007030,0.022495,-0.012176,18.66,3
...,...,...,...,...,...,...,...,...
349,MAIN,2023-05-04,2023-05-05,27.241206,0.013691,0.021022,40.33,1
350,MAIN,2023-02-23,2023-02-21,26.873292,0.033920,-0.022840,39.36,-2
351,MAIN,2023-02-23,2023-02-22,26.873292,0.033920,0.011433,39.81,-1
352,MAIN,2023-02-23,2023-02-23,26.873292,0.033920,0.009545,40.19,0
